In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "5"
os.environ["OPENBLAS_NUM_THREADS"] = "5"
os.environ["NUMEXPR_NUM_THREADS"] = "5"

# Example behaviour framework

In [2]:
from importlib.resources import files
import pandas as pd
import numpy as np
from lepto.gui.framework.glm_framework import create_offset, create_graph_matrix, create_categories, GLMFrameworkBehaviour
from lepto.gui.framework.data_preprocessor import DataPreprocessor
from lepto.standard.model.transformers import _detect_cat_var

## Data

In [3]:
path = files("lepto.data") / "sample_behaviour.csv"
df = pd.read_csv(path)

In [4]:
X_start = df[["age", "income", "region", "promo", "loyalty", "price"]]
w = df['w']
y = df['y']
behaviour = df['price']

## Framework

### Init

In [5]:
# Launch Data preparation
var_for_model_static = ["age", "income", "region"]
var_for_model_dynamic = ["age", "promo", "loyalty"]
var_for_model = list(set(var_for_model_static) | set(var_for_model_dynamic))
behaviour_var = 'price'
preproc = DataPreprocessor(
            df=df,
            variables=var_for_model)
X = preproc.run()
X[behaviour_var] = behaviour

In [6]:
# Create default values
transformer_data = preproc.transformer_data
## Variables types and categories
geographical_data_dict = {}
variable_types = {"age": "continuous", "income": "continuous", "region": "categorical", "promo": "continuous", "loyalty": "continuous"}
categories_list = create_categories(transformer_data, variable_types, geographical_data_dict)
categories_dict = dict(zip(transformer_data.var_num + transformer_data.var_cat, categories_list))
## Default offset
default_offset = create_offset(transformer_data, categories_list)
## Default adjacendy matrix
categories_var = transformer_data._get_categories_var()
adj = create_graph_matrix(variable_types, categories_var, geographical_data_dict)


In [7]:
# Split initialize values for static and elasticity model
categories_dict_static = {k: categories_dict[k] for k in _detect_cat_var(X[var_for_model_static]) if k in  categories_dict} 
categories_dict_dynamic = {k: categories_dict[k] for k in _detect_cat_var(X[var_for_model_dynamic]) if k in  categories_dict} 
categories_list_static = [values for values in categories_dict_static.values()]
categories_list_dynamic = [values for values in categories_dict_dynamic.values()]
default_offset_static = {k: default_offset[k] for k in var_for_model_static if k in  default_offset}
default_offset_dynamic = {k: default_offset[k] for k in var_for_model_dynamic if k in  default_offset}
adj_static = {k: adj[k] for k in var_for_model_static if k in  adj}
adj_dynamic = {k: adj[k] for k in var_for_model_dynamic if k in  adj}

### Launch

In [8]:
glm = GLMFrameworkBehaviour(
            var_for_model_static,
            var_for_model_dynamic,
            behaviour_var,
            monotone_shape='increasing',
            lam_grid=np.linspace(1e-2, 10, 2),
            lam_behaviour=0,
            adj_matrix_static=adj_static,
            offset_betas_static=default_offset_static,
            adj_matrix_dynamic=adj_dynamic,
            offset_betas_dynamic=default_offset_dynamic,
            categories_static=categories_list_static,
            categories_dynamic=categories_list_dynamic)

glm.fit(X, y, w)

In [9]:
best_glm = glm.grid_search.best_estimator_

In [10]:
best_glm.plot(X, y, w, "age", "static")